# 25 | LangChain Middleware：自定义拦截逻辑背后的request和handler

写自定义 Middleware 时，经常会看到这两个参数：

```python
@wrap_tool_call
def audit_tool(request, handler):
    ...
```

```python
@wrap_model_call
def switch_model(request, handler):
    ...
```

看起来很像“框架塞进来的两个神秘参数”。其实不神秘。

`request` 是这一次模型调用或工具调用的结构化请求；`handler` 是“把请求继续交给下游执行”的函数。

换成人话：`request` 像工单，里面写着这次要干什么；`handler` 像放行按钮，你按了，请求才会继续往下走。

## 一、为什么要单独讲它们

如果只会照着写 `handler(request)`，Middleware 很容易变成玄学：复制时能跑，稍微一改就开始看天吃饭。

真正理解 `request` 和 `handler` 后，你会知道 Middleware 为什么能做这些事：

- 调用前检查参数，发现风险直接拦住；
- 调用前改参数，比如切模型、改工具参数；
- 调用失败后重试；
- 调用后记录日志、统计耗时；
- 干脆不调用原逻辑，直接返回一个结果。

这篇不追求花活，只把这两个参数讲清楚。

## 二、先准备一个真实一点的工具

场景还是订单报表导出。

用户说：“帮我导出 20 万行订单数据”。Agent 可以调用工具，但工具执行前，我们希望 Middleware 先看一眼：是不是太大了？要不要拦下来？别让系统一上来就烤服务器。

In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import ToolMessage

load_dotenv(dotenv_path=".env")

# 云端模型用于演示工具调用。这里通过 OpenAI-compatible 接口接入。
cloud_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# 本地模型先准备好，后面演示 request.override(model=...) 时会用到。
local_model = init_chat_model("ollama:qwen2.5:14b")

In [ ]:
class ExportTaskInput(BaseModel):
    report_name: str = Field(description="报表名称，例如 order_export")
    file_format: Literal["xlsx", "csv"] = Field(description="导出格式")
    estimated_rows: int = Field(description="预计导出的数据行数")
    reason: str = Field(description="导出原因，例如月度财务结算")


@tool(args_schema=ExportTaskInput)
def create_export_task(
    report_name: str,
    file_format: Literal["xlsx", "csv"],
    estimated_rows: int,
    reason: str,
) -> dict:
    """创建订单报表导出任务。"""
    print(f"[工具执行] 创建导出任务：{report_name}, {file_format}, {estimated_rows} 行")
    return {
        "task_id": "EXP-2026-001",
        "status": "created",
        "report_name": report_name,
        "file_format": file_format,
        "estimated_rows": estimated_rows,
        "reason": reason,
    }

## 三、工具调用里的 request：这次要调用哪个工具

`@wrap_tool_call` 包住的是“一次工具调用”。

所以这里的 `request` 重点看：

- `request.tool_call["name"]`：这次要调用哪个工具；
- `request.tool_call["args"]`：模型准备传给工具的参数；
- `request.tool_call["id"]`：这次工具调用的 ID；
- `request.tool`：被调用的工具对象；
- `request.state / request.runtime`：当前 Agent 状态和运行时信息，后面文章再专门讲。

下面这个 Middleware 做两件事：先打印工具请求；如果导出超过 10 万行，就不让工具真的执行。

In [ ]:
@wrap_tool_call
def inspect_and_guard_tool(request, handler):
    # request.tool_call 是模型生成的工具调用请求，里面有工具名、参数和调用 ID。
    tool_name = request.tool_call["name"]
    args = request.tool_call.get("args", {})
    tool_call_id = request.tool_call["id"]

    print("[tool request]")
    # 为了看清结构，这里打印完整 request。生产日志不要直接这样打，里面可能有用户消息和业务参数。
    print(f"tool-request的完整内容:{request}")
    print(f"name={tool_name}")
    print(f"id={tool_call_id}")
    print(f"args={args}")
    print("=" * 40)

    # 这里选择“不调用 handler”，所以 create_export_task 不会真正执行。
    # 但仍然返回 ToolMessage，让 Agent 知道工具侧给出了什么结果。
    if tool_name == "create_export_task" and args.get("estimated_rows", 0) > 100_000:
        return ToolMessage(
            content="预计导出行数超过 100000，需要先走人工审批或拆分导出范围。",
            tool_call_id=tool_call_id,
        )

    # 调用 handler(request)，才表示继续执行原本的工具调用。
    return handler(request)

这里最关键的是：**不调用 `handler`，原工具就不会执行。**

这不是“工具执行失败”，而是 Middleware 主动接管了这次工具调用。比如审批、权限、危险操作拦截，都可以这样做。

## 四、模型调用里的 request：这次要把什么发给模型

`@wrap_model_call` 包住的是“一次模型调用”。

所以这里的 `request` 重点看：

- `request.messages`：这次要发给模型的消息；
- `request.model`：这次准备调用哪个模型；
- `request.tools`：这次模型可用的工具列表；
- `request.model_settings`：模型调用参数；
- `request.state / request.runtime`：当前 Agent 状态和运行时信息。

下面先做一个只打印、不改动的版本。

In [ ]:
def get_tool_names(tools):
    """把工具对象或工具 schema 整理成名称，方便打印观察。"""
    names = []
    for item in tools or []:
        if isinstance(item, dict):
            names.append(item.get("name") or item.get("function", {}).get("name") or "dict_tool")
        else:
            names.append(getattr(item, "name", item.__class__.__name__))
    return names


@wrap_model_call
def inspect_model_request(request, handler):
    # request.messages 是这次真正准备发给模型的消息列表。
    print("[model request]")
    # 为了看清结构，这里打印完整 request。生产日志不要直接这样打，里面可能有用户消息和运行时信息。
    print(f"model-request的完整内容:{request}")
    print(f"messages={len(request.messages)}")
    print(f"model={request.model.__class__.__name__}")
    print(f"tools={get_tool_names(request.tools)}")
    print("=" * 40)

    # 不改请求，只是继续调用模型。
    return handler(request)

把两个 Middleware 放进 Agent 里跑一次。

如果模型决定调用 `create_export_task`，你会先看到模型请求日志，再看到工具请求日志。工具被拦住后，Agent 还会继续根据工具返回的信息组织回复。

In [ ]:
debug_agent = create_agent(
    model=cloud_model,
    tools=[create_export_task],
    middleware=[
        inspect_model_request,
        inspect_and_guard_tool,
    ],
    system_prompt=(
        "你是订单报表导出助手。"
        "用户明确要求创建导出任务时，需要调用 create_export_task。"
        "如果工具提示需要审批或拆分导出范围，请直接向用户说明原因。"
    ),
)

result = debug_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "帮我创建一个订单报表导出任务，格式 xlsx，预计 200000 行，原因是月度财务结算。",
        }
    ]
})

print("\n[最终回复]")
print(result["messages"][-1].content)

这个输出可以重点看三件事：

1. 第一次 `[model request]`：Agent 准备把用户消息、系统提示和工具列表发给模型，让模型决定要不要调用工具；
2. `[tool request]`：模型已经生成了工具调用，Middleware 在工具真正执行前拿到了工具名和参数；
3. 第二次 `[model request]`：工具返回结果后，Agent 再把新的上下文发给模型，让模型组织最终回复。

所以 Middleware 不是“站在外面看热闹”，它真的卡在模型调用和工具调用的路上。

### 怎么看完整的 `model request`

你打印出来的 `ModelRequest(...)` 很长，先看这几块就够了：

| 字段 | 怎么理解 |
|---|---|
| `model=ChatOpenAI(...)` | 这次准备调用的模型对象。里面能看到 `model_name='deepseek-v4-flash'`、`openai_api_base=...`。`openai_api_key=SecretStr('**********')` 说明密钥被隐藏了，这是正常的。 |
| `messages=[HumanMessage(...)]` | 这次要发给模型的消息。第一次调用里只有用户问题，所以你看到 `messages=1`。 |
| `system_message=SystemMessage(...)` | `create_agent` 里配置的系统提示。它不是用户消息，但会参与本次模型调用。 |
| `tool_choice=None` | 没有强制模型必须调用哪个工具，模型可以自己决定。 |
| `tools=[StructuredTool(...)]` | 这次模型可用的工具列表。这里的 `StructuredTool` 里带着 `args_schema=ExportTaskInput`，也就是工具参数结构。 |
| `response_format=None` | 没有要求模型按某个结构化格式输出。 |
| `state={'messages': [...]}` | Agent 当前状态快照。这里暂时只需要知道它保存了当前对话消息。 |
| `runtime=Runtime(...)` | 运行时信息，比如 task id、stream writer、checkpoint 相关信息等。这个更偏 Agent 运行机制，下一篇再细讲。 |
| `model_settings={}` | 本次没有额外传模型参数。如果设置了 temperature、max_tokens 之类，通常会体现在这里。 |

也就是说，`model request` 回答的是：**这一次模型调用，准备用哪个模型、带哪些消息、给它哪些工具、附带哪些调用设置。**

### 怎么看完整的 `tool request`

`ToolCallRequest(...)` 更贴近工具执行现场，重点更明确：

| 字段 | 怎么理解 |
|---|---|
| `tool_call={'name': ..., 'args': ..., 'id': ..., 'type': ...}` | 模型刚刚生成的工具调用。这里最重要，能看到它要调用 `create_export_task`，并且传入 `estimated_rows=200000`。 |
| `tool=StructuredTool(...)` | 真正准备被执行的 Python 工具对象。里面能看到工具描述、参数 schema 和对应函数。 |
| `state={'messages': [...]}` | 工具调用发生时的 Agent 状态。这里通常已经包含用户消息，以及模型刚刚生成的带 tool call 的 `AIMessage`。 |
| `runtime=ToolRuntime(...)` | 工具执行阶段的运行时信息。比如工具调用上下文、状态访问能力等。 |

这里有一个很关键的细节：你没有看到 `[工具执行] 创建导出任务...`，说明 `create_export_task` 没有真正执行。

原因是 `inspect_and_guard_tool` 判断 `estimated_rows > 100_000` 后，直接返回了 `ToolMessage`，没有调用 `handler(request)`。

这正是 Middleware 的价值：工具还没动手，Middleware 先把风险拦住了。否则 20 万行导出任务可能已经创建成功，后面再说“我其实不想导了”，就晚了一步。

### 这次输出说明了什么

这次链路可以按顺序理解：

1. 用户提出导出需求；
2. Agent 进入第一次模型调用，`model request` 里带着用户消息、系统提示和工具列表；
3. 模型决定调用 `create_export_task`，并生成工具参数；
4. Agent 进入工具调用，`tool request` 里出现 `estimated_rows=200000`；
5. Middleware 判断数据量过大，直接返回审批提示，没有继续执行工具；
6. Agent 再调用一次模型，让模型根据工具返回的信息组织最终回复。

所以 `request` 不是一个抽象概念，它就是每次调用发生前，Middleware 能拿到的那份“现场记录”。

补一句工程上的提醒：完整 `request` 很适合调试和写文章截图，但生产日志里不要直接全量打印。它可能包含用户消息、工具参数、运行时信息；真要记录，也应该挑字段、做脱敏。


## 五、handler：继续、阻断、重试、改请求

`handler` 可以简单理解成：**继续执行下一步**。

但它不只是 `return handler(request)` 这么一种用法。

In [ ]:
@wrap_tool_call
def allow_tool(request, handler):
    # 最普通的写法：什么都不改，直接放行。
    return handler(request)


@wrap_tool_call
def block_big_export(request, handler):
    args = request.tool_call.get("args", {})

    # 不调用 handler，表示这次工具调用到这里就结束。
    if args.get("estimated_rows", 0) > 100_000:
        return ToolMessage(
            content="数据量太大，需要先审批。",
            tool_call_id=request.tool_call["id"],
        )

    return handler(request)

In [ ]:
@wrap_model_call
def retry_local_then_cloud(request, handler):
    # 先用本地模型试 3 次，适合处理 Ollama 服务偶尔没起来的情况。
    for attempt in range(1, 4):
        try:
            print(f"[模型调用] 本地 qwen2.5 第 {attempt} 次尝试")
            return handler(request.override(model=local_model))
        except Exception as e:
            print(f"[模型调用] 本地模型失败：{type(e).__name__}")

    # request.override(...) 会创建一个“改过的新请求”。
    # 这里只改 model，messages/tools 等其他信息仍然沿用原请求。
    print("[模型调用] 本地模型连续失败，切换到云端 deepseek-v4-flash")
    return handler(request.override(model=cloud_model))

这几个写法背后的逻辑很统一：

| 写法 | 含义 |
|---|---|
| `handler(request)` | 把当前请求继续交给下游执行 |
| 不调用 `handler` | Middleware 直接接管，本次调用不再继续 |
| 多次调用 `handler` | 可以做重试，但要注意成本和副作用 |
| `handler(request.override(...))` | 改掉这次请求的一部分，再交给下游执行 |
| 调用前后包一层逻辑 | 做日志、耗时统计、审计、兜底 |

尤其要注意工具调用：如果工具会发邮件、扣款、删文件，多次调用 `handler` 可能真的会执行多次。重试不是免费的，有时候还挺贵。

## 六、别和 state、runtime 混在一起

`request / handler` 只出现在 `wrap_model_call` 和 `wrap_tool_call` 这类 wrap-style hook 里。

它们关注的是：**这一次调用怎么处理**。

而 `state / runtime` 更像 Agent 当前运行过程里的上下文信息，适合做状态读取、上下文控制、跨步骤信息传递。这个会放到下一篇单独讲。

| 概念 | 更像什么 | 常见用途 |
|---|---|---|
| `request` | 本次调用的工单 | 看消息、工具名、参数、模型、设置 |
| `handler` | 继续执行按钮 | 放行、阻断、重试、改请求后再执行 |
| `state` | Agent 当前状态 | 看历史消息、维护自定义状态 |
| `runtime` | 本次运行环境 | 读取上下文、配置、运行时信息 |

记住一句就够了：

**request 决定“这次调用带着什么来”，handler 决定“这次调用还要不要继续往下走”。**

## 小结

`request` 和 `handler` 是自定义 Middleware 的核心入口。

工具侧的 `request` 能看到工具名、参数和调用 ID，所以适合做审批、权限、审计、参数检查。

模型侧的 `request` 能看到 messages、model、tools、model settings，所以适合做模型切换、重试、降级、成本控制。

`handler` 则负责把请求继续交给下游。你调用它，后面的调用继续；你不调用它，Middleware 就接管了这次调用。

下一篇再看 `state` 和 `runtime`。这两个不像 `request / handler` 这么“拦路”，但更适合拿来理解 Agent 正在经历什么。